# Goal of this notebook
Here we aim to gain superficial knowledge about node-level and graph-level features in different datasets.
For "dataset" we mean the ensamble of uniprot's node features and stringdb's connectivity.
### Used datasets
Main experiments:
- Alzheimer's Disease (AD)
- Parkinson's Disease (PD) 

Testing and validation:
- Amyotrophic Lateral Sclerosis (ALS)
- Huntington's Disease (HD)
- Frontotemporal Dementia (FTD)
- Multiple System Atrophy (MSA)

Diabetes (Type 1 and Type 2) are discarded fro experiments because they are too overlapping.
### Extracted features
Node-level feature:
- degree statistics
- centrality measures
- GO statistics
Graph level features:
- mean connectivity
- number of nodes connected components
- number of nodes and egdes per class


In [2]:
import pandas as pd
from pathlib import Path

source_path = Path('./sources')
test_source_path = source_path / Path('test_db')
diabetes_source_path = source_path / Path('diabetes')

AD_PATH = source_path / Path('UniProtKB-Alzheimer.tsv')
PD_PATH = source_path / Path('UniProtKB-Parkinson.tsv')
D1_PATH = test_source_path / Path('UniProtKB-MS.tsv')
D2_PATH = test_source_path / Path('UniProtKB-SLE.tsv')

ad_csv = pd.read_csv(AD_PATH, sep='\t')
pd_csv = pd.read_csv(PD_PATH, sep='\t')
t1_csv = pd.read_csv(D1_PATH, sep='\t')
t2_csv = pd.read_csv(D2_PATH, sep='\t')

entry_set_ad = set(ad_csv['Entry'].to_list())
entry_set_pd = set(pd_csv['Entry'].to_list())
entry_set_t1 = set(t1_csv['Entry'].to_list())
entry_set_t2 = set(t2_csv['Entry'].to_list())

print('C1 len: ', len(entry_set_t1))
print('C2 len: ', len(entry_set_t2))

common_test = set.intersection(entry_set_t1, entry_set_t2)
print('Common test_db proteins', len(common_test), "\n", common_test)

entry_set_neuro = set.union(entry_set_pd, entry_set_ad)
entry_set_diabetes = set.union(entry_set_t1, entry_set_t2)

print("neurodegenerative proteins: ", len(entry_set_neuro))
print("diabetes proteins: ", len(entry_set_diabetes))

common = set.intersection(entry_set_diabetes, entry_set_neuro)
print("common proteins: ", len(common), "\n", common)

print("common of commons: ", len(set.intersection(common_test, common)), "\n", set.intersection(common_test, common))

ad_csv.head()

C1 len:  89
C2 len:  91
Common test_db proteins 6 
 {'P06702', 'P05109', 'Q8IZA0', 'Q13568', 'P01911', 'Q9HAT2'}
neurodegenerative proteins:  390
diabetes proteins:  174
common proteins:  11 
 {'P06702', 'Q00535', 'Q13501', 'P48023', 'P20700', 'Q99497', 'P68036', 'P10636', 'P55290', 'P09429', 'P49768'}
common of commons:  1 
 {'P06702'}


,Entry,Entry Name,Protein names,Gene Names,Gene Ontology (biological process),Gene Ontology (cellular component),Gene Ontology (molecular function),Gene Ontology IDs,Gene Ontology (GO),STRING
0,A1KXE4,F168B_HUMAN,Myelin-associated neurite-outgrowth inhibitor ...,FAM168B KIAA0280L MANI,NaN,axon [GO:0030424]; extracellular exosome [GO:0...,NaN,GO:0005886; GO:0030424; GO:0048471; GO:0070062,axon [GO:0030424]; extracellular exosome [GO:0...,9606.ENSP00000374565
1,A4D1B5,GSAP_HUMAN,Gamma-secretase-activating protein (GSAP) (Pro...,GSAP PION,positive regulation of amyloid-beta formation ...,trans-Golgi network [GO:0005802],amyloid-beta binding [GO:0001540],GO:0001540; GO:0005802; GO:0030162; GO:1902004,trans-Golgi network [GO:0005802]; amyloid-beta...,9606.ENSP00000257626
2,O00189,AP4M1_HUMAN,AP-4 complex subunit mu-1 (AP-4 adaptor comple...,AP4M1 MUARP2,autophagosome assembly [GO:0000045]; Golgi to ...,AP-4 adaptor complex [GO:0030124]; clathrin ad...,protein domain specific binding [GO:0019904]; ...,GO:0000045; GO:0005769; GO:0005802; GO:0005829...,AP-4 adaptor complex [GO:0030124]; clathrin ad...,9606.ENSP00000403663
3,O00213,APBB1_HUMAN,Amyloid beta precursor protein binding family ...,APBB1 FE65 RIR,apoptotic process [GO:0006915]; axonogenesis [...,cytoplasm [GO:0005737]; endoplasmic reticulum ...,amyloid-beta binding [GO:0001540]; chromatin b...,GO:0000122; GO:0001540; GO:0003682; GO:0003713...,cytoplasm [GO:0005737]; endoplasmic reticulum ...,9606.ENSP00000477213
4,O00429,DNM1L_HUMAN,Dynamin-1-like protein (EC 3.6.5.5) (Dnm1p/Vps...,DNM1L DLP1 DRP1,calcium ion transport [GO:0006816]; endocytosi...,brush border [GO:0005903]; clathrin-coated pit...,GTP binding [GO:0005525]; GTP-dependent protei...,GO:0000266; GO:0003924; GO:0005096; GO:0005525...,brush border [GO:0005903]; clathrin-coated pit...,9606.ENSP00000449089


Notes from the analysis below:
- Every couple of ALS, HD and MSA are suitable because they share few proteins
- Those share few proteins with alzheimer and parkinson's too.

TODO: graph analyse those

In [3]:
from data_visualization.graph_analyzer import GraphAnalyzer
from data_pipeline.dataset import NeuroDegAnc2VecDataset
import torch_geometric.transforms as T
from data_pipeline import *

import pandas as pd
from pathlib import Path

import itertools

project_root = Path('.')
source_path = project_root / 'sources'
test_source_path = source_path / 'test_db'
diabetes_source_path = source_path / 'diabetes'


ad_path  = source_path / 'UniProtKB-Alzheimer.tsv'
pd_path  = source_path / 'UniProtKB-Parkinson.tsv'
als_path = test_source_path / 'UniProtKB-ALS.tsv'   # Amyotrophic Lateral Sclerosis
hd_path  = test_source_path / 'UniProtKB-HD.tsv'    # Huntington's Disease
ftd_path = test_source_path / 'UniProtKB-FTD.tsv'   # Frontotemporal Dementia
msa_path = test_source_path / 'UniProtKB-MSA.tsv'   # Multiple System Atrophy

to_compare_ds = [ad_path, pd_path, als_path, hd_path, ftd_path, msa_path]

def short_name_extract(s):
    s = str(s)
    return s.split('-')[-1].split('.')[0]

for i in range(len(to_compare_ds)):
    d1 = to_compare_ds[i]
    ds1 = pd.read_csv(d1, sep='\t')
    string_id_1 = set(ds1['STRING'].to_list())
    for j in range(i+1, len(to_compare_ds)):
        d2 = to_compare_ds[j]
        ds2 = pd.read_csv(d2, sep='\t')
        string_id_2 = set(ds2['STRING'].to_list())

        common = set.intersection(string_id_1, string_id_2)
        print(f'Common prots: {short_name_extract(d1)}({len(string_id_1)}) {short_name_extract(d2)}({len(string_id_2)}) | {len(common)} | {common}')
        





/home/emanuele/Documents/Tesi/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Common prots: Alzheimer(231) Parkinson(173) | 20 | {'9606.ENSP00000366506', '9606.ENSP00000284981', '9606.ENSP00000358513', '9606.ENSP00000419782', '9606.ENSP00000345023', '9606.ENSP00000309457', '9606.ENSP00000298910', '9606.ENSP00000366047', '9606.ENSP00000377296', '9606.ENSP00000264710', '9606.ENSP00000328088', '9606.ENSP00000340820', '9606.ENSP00000354687', '9606.ENSP00000500990', '9606.ENSP00000301015', nan, '9606.ENSP00000309124', '9606.ENSP00000284440', '9606.ENSP00000311816', '9606.ENSP00000374455'}
Common prots: Alzheimer(231) ALS(101) | 1 | {nan}
Common prots: Alzheimer(231) HD(62) | 0 | set()
Common prots: Alzheimer(231) FTD(41) | 0 | set()
Common prots: Alzheimer(231) MSA(109) | 1 | {nan}
Common prots: Parkinson(173) ALS(101) | 1 | {nan}
Common prots: Parkinson(173) HD(62) | 0 | set()
Common prots: Parkinson(173) FTD(41) | 0 | set()
Common prots: Parkinson(173) MSA(109) | 1 | {nan}
Common prots: ALS(101) HD(62) | 4 | {'9606.ENSP00000368022;', '9606.ENSP00000264867;', '9606.

In [29]:
def get_unique_go_terms(df, column_name='Gene Ontology IDs'):
    """Estrae l'insieme dei termini GO unici da una colonna di stringhe separate da ';'."""
    # 1. Splitta ed elimina gli spazi usando la regex `;\s*`
    splitted_series = df[column_name].fillna("").str.split(r';\s*')
    
    # 2. Srotola la lista di liste (flattening) in un unico set, escludendo stringhe vuote
    unique_terms = set(
        term for sublist in splitted_series 
        for term in sublist if term.strip() != ""
    )
    return unique_terms

# --- CALCOLO DELL'INDICE DI JACCARD ---
print('===== Pairwise Jaccard comparison =====')
to_compare_ds = [ad_path, pd_path, als_path, hd_path, ftd_path, msa_path]
for i in range(len(to_compare_ds)):
    d1 = to_compare_ds[i]
    ds1 = pd.read_csv(d1, sep='\t')
    go_set_1 = get_unique_go_terms(ds1)
    for j in range(i+1, len(to_compare_ds)):
        d2 = to_compare_ds[j]
        ds2 = pd.read_csv(d2, sep='\t')
        go_set_2 = get_unique_go_terms(ds2)

        # Calcolo di Intersezione e Unione
        intersezione = go_set_1.intersection(go_set_2)
        unione = go_set_1.union(go_set_2)

        # Indice di Jaccard
        jaccard_index = len(intersezione) / len(unione) if len(unione) > 0 else 0
        print(f"J({short_name_extract(d1)}, {short_name_extract(d2)}) = {jaccard_index:.3f}")

print("\n")
print("===== Train-Test Jaccard Comparison =====")

def merge_go_set(df_path1, df_path2, column_name='Gene Ontology IDs') -> set:
    go1 = get_unique_go_terms(pd.read_csv(df_path1, sep='\t'), column_name)
    go2 = get_unique_go_terms(pd.read_csv(df_path2, sep='\t'), column_name)

    return set.union(go1, go2)

ad_pd = merge_go_set(ad_path, pd_path)
als_hd = merge_go_set(als_path, hd_path)
als_msa = merge_go_set(als_path, msa_path)
hd_msa = merge_go_set(hd_path, msa_path)

for k,v in {'ALS-HD': als_hd, 'ALS-MSA': als_msa, 'HD-MSA': hd_msa}.items():
    intersection = set.intersection(ad_pd, v)
    union = set.union(ad_pd, v)
    jaccard_index = len(intersection) / len(union) if len(union) > 0 else 0
    print(f"J(AD-PD, {k}) = {jaccard_index:.3f}")
        

===== Pairwise Jaccard comparison =====
J(Alzheimer, Parkinson) = 0.332
J(Alzheimer, ALS) = 0.283
J(Alzheimer, HD) = 0.182
J(Alzheimer, FTD) = 0.228
J(Alzheimer, MSA) = 0.267
J(Parkinson, ALS) = 0.316
J(Parkinson, HD) = 0.192
J(Parkinson, FTD) = 0.228
J(Parkinson, MSA) = 0.250
J(ALS, HD) = 0.222
J(ALS, FTD) = 0.444
J(ALS, MSA) = 0.297
J(HD, FTD) = 0.224
J(HD, MSA) = 0.196
J(FTD, MSA) = 0.258


===== Train-Test Jaccard Comparison =====
J(AD-PD, ALS-HD) = 0.319
J(AD-PD, ALS-MSA) = 0.349
J(AD-PD, HD-MSA) = 0.295
